In [ ]:
import networkx as nx

In [ ]:
G = nx.Graph()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Cancer Evolution Arena"
DATA_DIR = f"{PROJECT_DIR}/Data"
NOTEBOOK_DIR = f"{PROJECT_DIR}/Notebooks"

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(NOTEBOOK_DIR, exist_ok=True)

In [ ]:
!cp luad_tcga_pub.tar.gz "/content/drive/MyDrive/Cancer Evolution Arena/Data/"

cp: cannot stat 'luad_tcga_pub.tar.gz': No such file or directory


In [ ]:
!tar -xzf "/content/drive/MyDrive/Cancer Evolution Arena/Data/luad_tcga_pub.tar.gz" -C "/content/drive/MyDrive/Cancer Evolution Arena/Data/"

In [ ]:
import pandas as pd

mutation_path = "/content/drive/MyDrive/Cancer Evolution Arena/Data/luad_tcga_pub/data_mutations.txt"

df = pd.read_csv(
    mutation_path,
    sep="\t",
    comment="#",
    low_memory=False
)

patient_gene = pd.crosstab(
    df["Tumor_Sample_Barcode"],
    df["Hugo_Symbol"]
)

patient_gene = (patient_gene > 0).astype(int)

print(patient_gene.shape)

(230, 15130)


In [ ]:
patient_gene.to_csv("/content/drive/MyDrive/Cancer Evolution Arena/Data/patient_gene_matrix.csv")

In [ ]:
patient_gene = pd.read_csv(
    "/content/drive/MyDrive/Cancer Evolution Arena/Data/patient_gene_matrix.csv",
    index_col=0
)

In [ ]:
import pandas as pd

sig_pairs = pd.read_csv(
    "/content/drive/MyDrive/Cancer Evolution Arena/Data/significant_pairs.csv"
)

sig_pairs.head()

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
0,ABCA12,ABCC11,6,5.571429,0.004708,0.032761
1,ABCA12,FLG2,9,4.000000,0.005346,0.035508
2,ABCA12,LPA,6,4.825000,0.008017,0.043854
3,ABCA12,LRP2,8,4.529231,0.004086,0.030406
4,ABCA12,LRRIQ1,6,4.825000,0.008017,0.043854


In [ ]:
for _, row in sig_pairs.iterrows():

    G.add_edge(
        row["GeneA"],
        row["GeneB"],
        weight=row["OddsRatio"]
    )

In [ ]:
print(
    "Nodes:",
    G.number_of_nodes()
)

print(
    "Edges:",
    G.number_of_edges()
)

Nodes: 325
Edges: 10374


In [ ]:
!pip install python-louvain

In [ ]:
import community as community_louvain

In [ ]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

communities = greedy_modularity_communities(G)

print("Number of communities:", len(communities))

for i, community in enumerate(communities):
    print(f"Community {i}: {len(community)} genes")
    print(sorted(list(community))[:30])
    print()

Number of communities: 5
Community 0: 150 genes
['ABCA13', 'ACAN', 'ACTN2', 'ADAMTS16', 'ADAMTS20', 'ADCY2', 'AFF2', 'AHNAK2', 'ALK', 'ALPK2', 'ANK3', 'APOB', 'ATRNL1', 'ATRX', 'BAI3', 'BCLAF1', 'BLTP1', 'C1orf173', 'CACNA1C', 'CDH12', 'CDH18', 'CFH', 'CNTNAP5', 'COL14A1', 'COL19A1', 'COL22A1', 'COL3A1', 'COL6A3', 'CPED1', 'CPS1']

Community 1: 128 genes
['ABCB5', 'ADAMTS12', 'ADAMTS18', 'ADAMTS5', 'ADCY8', 'AHNAK', 'AKAP6', 'ANKRD30A', 'ARPP21', 'ASTN1', 'ASTN2', 'BOD1L1', 'BRAF', 'BRINP3', 'CACNA1E', 'CCDC178', 'CDH10', 'CDH9', 'CENPF', 'CNTNAP2', 'COL11A1', 'COL12A1', 'COL4A5', 'COL5A1', 'COL5A2', 'DMBT1', 'DMD', 'DNAH3', 'DNAH9', 'DOCK2']

Community 2: 38 genes
['ABCA12', 'ABCC11', 'ADAMTS2', 'ANK2', 'ASXL3', 'BCORL1', 'CLCN1', 'CNTN5', 'DCAF4L2', 'DNAH7', 'FAM47B', 'FLNC', 'GLI3', 'GRIN2A', 'HCN1', 'ITGAX', 'LCT', 'LRRIQ1', 'MROH2B', 'NALCN', 'NBPF10', 'NLRP12', 'OR2L13', 'PAPPA2', 'PCDHB3', 'PCNT', 'PLCL1', 'PXDNL', 'SALL1', 'SLITRK2']

Community 3: 6 genes
['ALMS1', 'ASPM', 'ATM

In [ ]:
community_rows = []

for i, community in enumerate(communities):
    for gene in community:
        community_rows.append({
            "Gene": gene,
            "Community": i
        })

community_df = pd.DataFrame(community_rows)

community_df.head()

,Gene,Community
0,DSCAM,0
1,ITGA8,0
2,SLC39A12,0
3,BLTP1,0
4,CUBN,0


In [ ]:
print("Density:", nx.density(G))

Density: 0.19703703703703704


In [ ]:
driver_genes = [
    "TP53", "KRAS", "EGFR", "STK11", "KEAP1",
    "NF1", "BRAF", "MET", "PIK3CA", "RB1",
    "ALK", "ROS1", "RET", "ERBB2", "SMARCA4"
]

driver_sig = sig_pairs[
    (sig_pairs["GeneA"].isin(driver_genes)) &
    (sig_pairs["GeneB"].isin(driver_genes))
].copy()

driver_sig

,GeneA,GeneB,CoOccurrence,OddsRatio,PValue,Adj_P
963,ALK,TP53,17,3.683333,0.007378,0.042672
4929,EGFR,KRAS,2,0.101287,0.000123,0.004675
4931,EGFR,STK11,0,0.000000,0.001123,0.015099
7244,KRAS,NF1,2,0.124266,0.000649,0.011287
7245,KRAS,STK11,21,2.783626,0.004966,0.033966
7246,KRAS,TP53,25,0.445122,0.007194,0.041900
8799,NF1,TP53,21,3.093023,0.006245,0.038162
10148,STK11,TP53,11,0.371408,0.008858,0.047063


In [ ]:
G_driver = nx.Graph()

for _, row in driver_sig.iterrows():
  G_driver.add_edge(
      row["GeneA"],
      row["GeneB"],
      weight=abs(row["OddsRatio"]),
      odds_ratio=row["OddsRatio"],
      adj_p=row["Adj_P"],
      cooccurrence=row["CoOccurrence"]
  )

print("Nodes:", G_driver.number_of_nodes())
print("Edges:", G_driver.number_of_edges())
print("Density:", nx.density(G_driver))

Nodes: 6
Edges: 8
Density: 0.5333333333333333


In [ ]:
print(list(G_driver.nodes()))

['ALK', 'TP53', 'EGFR', 'KRAS', 'STK11', 'NF1']


In [ ]:
for u, v, data in G_driver.edges(data=True):
    print(
        f"{u} -- {v}",
        "OR =", round(data["odds_ratio"], 2),
        "Adj_P =", round(data["adj_p"], 5)
    )

ALK -- TP53 OR = 3.68 Adj_P = 0.04267
TP53 -- KRAS OR = 0.45 Adj_P = 0.0419
TP53 -- NF1 OR = 3.09 Adj_P = 0.03816
TP53 -- STK11 OR = 0.37 Adj_P = 0.04706
EGFR -- KRAS OR = 0.1 Adj_P = 0.00467
EGFR -- STK11 OR = 0.0 Adj_P = 0.0151
KRAS -- NF1 OR = 0.12 Adj_P = 0.01129
KRAS -- STK11 OR = 2.78 Adj_P = 0.03397


In [ ]:
!find . -name "*clinical*"

./drive/MyDrive/Cancer Evolution Arena/Data/luad_tcga_pub/meta_clinical_sample.txt
./drive/MyDrive/Cancer Evolution Arena/Data/luad_tcga_pub/meta_clinical_patient.txt
./drive/MyDrive/Cancer Evolution Arena/Data/luad_tcga_pub/data_clinical_patient.txt
./drive/MyDrive/Cancer Evolution Arena/Data/luad_tcga_pub/data_clinical_sample.txt


In [ ]:
clinical_patient = pd.read_csv(
    "/content/drive/MyDrive/Cancer Evolution Arena/Data/luad_tcga_pub/data_clinical_patient.txt",
    sep="\t",
    comment="#"
)

print(clinical_patient.columns.tolist())
print(clinical_patient.head())

['PATIENT_ID', 'AGE', 'SEX', 'HISTOLOGICAL_SUBTYPE', 'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'PRETREATMENT_HISTORY', 'PRIMARY_TUMOR_PATHOLOGIC_SPREAD', 'PRIOR_DIAGNOSIS', 'RESIDUAL_TUMOR', 'TOBACCO_SMOKING_HISTORY_INDICATOR', 'OS_STATUS', 'OS_MONTHS', 'DFS_STATUS', 'DFS_MONTHS']
     PATIENT_ID AGE     SEX  \
0  TCGA-05-4249  67    Male   
1  TCGA-05-4382  68    Male   
2  TCGA-05-4384  66    Male   
3  TCGA-05-4389  70    Male   
4  TCGA-05-4390  58  Female   

                                HISTOLOGICAL_SUBTYPE ICD_O_3_HISTOLOGY  \
0  Lung Adenocarcinoma- Not Otherwise Specified (...            8140/3   
1                  Lung Adenocarcinoma Mixed Subtype            8255/3   
2                  Lung Adenocarcinoma Mixed Subtype            8255/3   
3                  Lung Adenocarcinoma Mixed Subtype            8255/3   
4                  Lung Adenocarcinoma Mixed Subtype            8255/3   

  ICD_O_3_SITE PRETREATMENT_HISTORY PRIMARY_TUMOR_PATHOLOGIC_SPREAD  \
0        C34.3      

In [29]:
print(patient_gene.index[:10])
print(clinical_patient["PATIENT_ID"].head(10))

Index(['TCGA-05-4249-01', 'TCGA-05-4382-01', 'TCGA-05-4384-01',
       'TCGA-05-4389-01', 'TCGA-05-4390-01', 'TCGA-05-4395-01',
       'TCGA-05-4396-01', 'TCGA-05-4398-01', 'TCGA-05-4402-01',
       'TCGA-05-4403-01'],
      dtype='object', name='Tumor_Sample_Barcode')
0    TCGA-05-4249
1    TCGA-05-4382
2    TCGA-05-4384
3    TCGA-05-4389
4    TCGA-05-4390
5    TCGA-05-4395
6    TCGA-05-4396
7    TCGA-05-4398
8    TCGA-05-4402
9    TCGA-05-4403
Name: PATIENT_ID, dtype: object
